# 1-) Creacion de Agentes con componentes core de LangChain

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print(f'OPENAI_API_KEY: {bool(os.getenv("OPENAI_API_KEY"))}')

OPENAI_API_KEY: True


In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini") # interfaz para distintos proveedores de LLM (anthropic, openai, google_genai)

response = llm.invoke('Que es LangChain? en una sentencia corta')

print(f'{"#"*30}\n{response.content}\n{"#"*30}')

##############################
LangChain es un marco de trabajo para construir aplicaciones de procesamiento de lenguaje natural que integran modelos de lenguaje, herramientas y fuentes de datos.
##############################


# 2-) Plantillas de Prompt **(instrucciones reutilizables)**

In [3]:
from langchain_core.prompts import PromptTemplate

 # plantillas dinamicas simples
prompt_simple = PromptTemplate(
    input_variables=["tema"],
    template="Hablame sobre este tema: {tema}",
)

prompt_formato = prompt_simple.format(tema="Python")

prompt_formato

'Hablame sobre este tema: Python'

In [4]:
from langchain_core.prompts import ChatPromptTemplate

# instrucciones basadas en ROLES: System, Human, Assistant
chat_template = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de IA especializado en Oracle, tu deber es responder acerca de temas relacionados con el EBS, Fussion, ERP, RPA, etc. del ecosistema oracle"),
    ("human", "Hola, quiero saber sobre: {topic}"),
])

messages = chat_template.format_messages(topic="Oracle Cloud")

for message in messages:
    print(f'[{message.type}]: {message.content}')

[system]: Eres un asistente de IA especializado en Oracle, tu deber es responder acerca de temas relacionados con el EBS, Fussion, ERP, RPA, etc. del ecosistema oracle
[human]: Hola, quiero saber sobre: Oracle Cloud


# 3-) Chains (Encadenar y limpiar la salida)

In [5]:
from langchain_core.output_parsers import StrOutputParser

chain = chat_template | llm | StrOutputParser()

print(f"Chain: {chain.invoke({"topic": "Oracle Cloud Infrastructure"})}")

Chain: Oracle Cloud Infrastructure (OCI) es una plataforma de computación en la nube que ofrece un conjunto amplio de servicios que permiten a las empresas crear, desplegar y gestionar aplicaciones y servicios en la nube. OCI se caracteriza por su enfoque en el rendimiento, la seguridad y la escalabilidad. A continuación, te presento algunos aspectos clave de Oracle Cloud Infrastructure:

1. **Servicios de Cómputo**: OCI ofrece instancias de cómputo flexibles que pueden ser configuradas con diferentes cantidades de recursos de CPU y memoria. Esto incluye instancias estándar, de alta disponibilidad y optimizadas para cargas de trabajo específicas, como machine learning o procesamiento gráfico.

2. **Almacenamiento**: La plataforma proporciona diversas opciones de almacenamiento, incluyendo almacenamiento en bloque, almacenamiento de objetos y almacenamiento de archivos. Esto permite a los usuarios elegir la mejor solución para sus necesidades de datos.

3. **Redes**: OCI incluye opcione

# 4-) Memoria y Contexto

In [20]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

memory = InMemoryChatMessageHistory()

# contexto que deba recordar el chat
memory.add_message(HumanMessage(content="Hello, im Shawn. How are you?"))
memory.add_message(AIMessage(content="I'm doing well Shawn, thank you for asking!"))

memory.add_message(HumanMessage(content="What's your name?"))
memory.add_message(AIMessage(content="My name is LangChain."))

memory.add_message(HumanMessage(content="What's your favorite color?"))
memory.add_message(AIMessage(content="My favorite color is blue."))

memory.add_message(HumanMessage(content="Mine is green!"))

In [7]:
for m in memory.messages:
    print(f'[{m.type}]: {m.content}')

[human]: Hello, im Shawn. How are you?
[ai]: I'm doing well Shawn, thank you for asking!
[human]: What's your name?
[ai]: My name is LangChain.
[human]: What's your favorite color?
[ai]: My favorite color is blue.
[human]: Mine is green!


In [8]:
chat_memoria = ChatPromptTemplate.from_messages([
    ("system", "Eres un agente de IA que puede responder preguntas."),
    ("placeholder", "{history}"),
    ("human", "{question}"),
])

chain_memoria = chat_memoria | llm | StrOutputParser()

print(memory.messages)

[HumanMessage(content='Hello, im Shawn. How are you?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm doing well Shawn, thank you for asking!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's your name?", additional_kwargs={}, response_metadata={}), AIMessage(content='My name is LangChain.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's your favorite color?", additional_kwargs={}, response_metadata={}), AIMessage(content='My favorite color is blue.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Mine is green!', additional_kwargs={}, response_metadata={})]


In [9]:
resultado = chain_memoria.invoke({
    "history": memory.messages, # contexto anterior
    "question": "What was my name again and my favorite color pls?" # debe recordar el contexto anterior
}) 

print(resultado)

Your name is Shawn, and your favorite color is green!


# 5-) Tools

In [10]:
from langchain_core.tools import tool

@tool
def multiply(x: int, y: int) -> int:
    """
    multiply(x: int, y: int) -> int
    Returns the product of x and y.
    """
    return x * y

@tool 
def add(x: int, y: int) -> int:
    """ 
    add(x: int, y: int) -> int
    Returns the sum of x and y.
    """
    return x + y 

@tool
def divide(x: int, y: int) -> int:
    """
    divide(x: int, y: int) -> int
    Returns the quotient of x and y.
    """
    return x / y 

In [ ]:
tools = [multiply, add, divide]

for tool in tools:
    print('#'*30)
    print(f'Tool Name: {tool.name}, Tool Description: {tool.description}')
    print('#'*30, '\n')

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini")
agent = create_agent(model=llm, tools=[multiply, add, divide])

In [12]:
result = agent.invoke({"messages": [{"role": "user", "content": "What 2 plus 2 equals?"}]})

In [ ]:
def pprint(messages):
    for msj in result['messages']:
        print(f"[{msj.type.upper()}]: {msj.content}")

pprint(result)

[HUMAN]: What 2 plus 2 equals?
[AI]: 
[TOOL]: 4
[AI]: 2 plus 2 equals 4.
